### Deep agent with real persistent stores and DB:

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")
os.environ["PG_DATABASE_URL"]=os.getenv("PG_DATABASE_URL")

#### Initialize model for the use:

In [3]:
from langchain_ollama import ChatOllama

model = ChatOllama(
    model="gemma4:e4b",
    temperature=0.3
)

response = model.invoke("Explain postgres SQL database and mongoDB DB.")

print(response.content)

This is a fantastic question because PostgreSQL and MongoDB represent the two major philosophical approaches to data storage today: **Structure vs. Flexibility.**

While both are powerful tools for storing and retrieving information, they operate on fundamentally different principles, which dictates what kind of applications they are best suited for.

Here is a detailed explanation of both databases, followed by a direct comparison.

---

## 🐘 1. PostgreSQL (The Structured Approach)

PostgreSQL is a powerful, open-source **Relational Database Management System (RDBMS)**. It is the gold standard for structured data and data integrity.

### 💡 How It Works: The Relational Model
PostgreSQL organizes data into **tables**. Think of a table like a highly organized spreadsheet.

*   **Tables:** Each table represents a specific entity (e.g., `Users`, `Products`, `Orders`).
*   **Rows:** Each row is a single record (e.g., one specific user).
*   **Columns:** Each column defines a specific attrib

In [4]:
from langchain_ollama import ChatOllama

model = ChatOllama(
    model="gemma4:e4b",
    temperature=0.3
)

### Configuring Postgres SQL DB for persistent store: 

### implementing chat isolation based on Thread_id (namespace)

In [5]:
import os
from dotenv import load_dotenv
from deepagents import create_deep_agent
from deepagents.backends import StoreBackend, CompositeBackend, StateBackend
from langgraph.store.postgres import PostgresStore

load_dotenv()
postgres_url = os.getenv("PG_DATABASE_URL")

# --- 6b. PostgresStore (open once, keep alive for the session) ---------------
_store_ctx = PostgresStore.from_conn_string(postgres_url)
postgres_store = _store_ctx.__enter__()
postgres_store.setup()   # creates langgraph store tables if they don't exist
print("PostgresStore initialised")

def make_backend(rt):
    return CompositeBackend(
        default=StateBackend(rt),           # ephemeral working files
        routes={
            "/memories/": StoreBackend(     # persistent, scoped by user+thread
                rt,
                store=postgres_store,
                namespace=lambda rt: (
                    rt.server_info.user.identity if rt.server_info 
                    else getattr(rt.context, "user_id", "default_local_user"),
                    
                    rt.execution_info.thread_id, 
                    "filesystem"
                ), 
            )
        },
    )

agent = create_deep_agent(
    model=model,
    backend=make_backend,
    store=postgres_store,
    system_prompt=(
        "You are a helpful assistant with persistent memory. "
        "Whenever the user shares personal information, preferences, or important facts, "
        "proactively save them to /memories/profile.txt so you can recall them later. "
        "Before answering questions about the user, read /memories/profile.txt first."
    ),
)

print("Agent created and ready")

PostgresStore initialised
Agent created and ready


In [6]:
from langchain_core.messages import HumanMessage

config = {
    "configurable": {
        "thread_id": "thread_45",
        # Inject user_id into context so make_namespace() can read it locally
        "context": type("Ctx", (), {"user_id": "rohit_45"})(),
    }
}

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Hi! My name is Rohit Sharma, I am indian cricket playet. I have won ICC world cup of 2007 and 2024 also won ICC CT of 2013 and 2025. My favorite cricketing shot is Pull shot."}]}, 
    config=config
)

print("Response:", result["messages"][-1].content)

c:\Users\gaura\OneDrive\Desktop\AI projects\agentic_ai_advanced\DeepAgents\deepagents\.venv\Lib\site-packages\deepagents\middleware\filesystem.py:1679: LangChainDeprecationWarning: Passing a callable (factory) as `backend` is deprecated and will be removed in deepagents==0.7.0. Pass a `BackendProtocol` instance directly instead (e.g. `StateBackend()`).
  backend = self._get_backend(request.runtime)  # ty: ignore[invalid-argument-type]
C:\Users\gaura\AppData\Local\Temp\ipykernel_26192\3982891131.py:18: LangChainDeprecationWarning: Passing `runtime` to `StateBackend` is deprecated and will be removed in deepagents==0.7.0. `StateBackend` now reads and writes state via `get_config()`. Use `StateBackend()` instead.
  default=StateBackend(rt),           # ephemeral working files
C:\Users\gaura\AppData\Local\Temp\ipykernel_26192\3982891131.py:20: LangChainDeprecationWarning: Passing `runtime` to `StoreBackend` is deprecated and will be removed in deepagents==0.7.0. `StoreBackend` now obtains 

Response: I have saved your details. I have noted that you are an Indian cricket player, have won the ICC World Cup in 2007 and 2024, and the ICC CT in 2013 and 2025. Your favorite shot is the Pull shot.


In [7]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Hi! Can you tell me about my profile?"}]}, 
    config=config
)

print("Response:", result["messages"][-1].content)

c:\Users\gaura\OneDrive\Desktop\AI projects\agentic_ai_advanced\DeepAgents\deepagents\.venv\Lib\site-packages\deepagents\middleware\filesystem.py:1679: LangChainDeprecationWarning: Passing a callable (factory) as `backend` is deprecated and will be removed in deepagents==0.7.0. Pass a `BackendProtocol` instance directly instead (e.g. `StateBackend()`).
  backend = self._get_backend(request.runtime)  # ty: ignore[invalid-argument-type]
C:\Users\gaura\AppData\Local\Temp\ipykernel_26192\3982891131.py:18: LangChainDeprecationWarning: Passing `runtime` to `StateBackend` is deprecated and will be removed in deepagents==0.7.0. `StateBackend` now reads and writes state via `get_config()`. Use `StateBackend()` instead.
  default=StateBackend(rt),           # ephemeral working files
C:\Users\gaura\AppData\Local\Temp\ipykernel_26192\3982891131.py:20: LangChainDeprecationWarning: Passing `runtime` to `StoreBackend` is deprecated and will be removed in deepagents==0.7.0. `StoreBackend` now obtains 

Response: Your profile contains the following information:

*   **Name:** Rohit Sharma
*   **Profession:** Indian cricket player
*   **Achievements:**
    *   ICC World Cup: 2007, 2024
    *   ICC CT: 2013, 2025
*   **Favorite Shot:** Pull shot


### Try with same user name but in different thread_id - Change in config

In [8]:
config = {
    "configurable": {
        "thread_id": "thread_264",
        # Inject user_id into context so make_namespace() can read it locally
        "context": type("Ctx", (), {"user_id": "rohit_45"})(),
    }
}

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Hi! Also i have won 6 IPL trophies in my cricketing career, 5 cups with Mumbai Indians frenchise as a captain and In 2009 i won as a player in Deccan Changers team of IPL."}]}, 
    config=config
)

print("Response:", result["messages"][-1].content)

c:\Users\gaura\OneDrive\Desktop\AI projects\agentic_ai_advanced\DeepAgents\deepagents\.venv\Lib\site-packages\deepagents\middleware\filesystem.py:1679: LangChainDeprecationWarning: Passing a callable (factory) as `backend` is deprecated and will be removed in deepagents==0.7.0. Pass a `BackendProtocol` instance directly instead (e.g. `StateBackend()`).
  backend = self._get_backend(request.runtime)  # ty: ignore[invalid-argument-type]
C:\Users\gaura\AppData\Local\Temp\ipykernel_26192\3982891131.py:18: LangChainDeprecationWarning: Passing `runtime` to `StateBackend` is deprecated and will be removed in deepagents==0.7.0. `StateBackend` now reads and writes state via `get_config()`. Use `StateBackend()` instead.
  default=StateBackend(rt),           # ephemeral working files
C:\Users\gaura\AppData\Local\Temp\ipykernel_26192\3982891131.py:20: LangChainDeprecationWarning: Passing `runtime` to `StoreBackend` is deprecated and will be removed in deepagents==0.7.0. `StoreBackend` now obtains 

Response: I have noted your cricketing achievements: 6 IPL trophies in your career, 5 cups as captain with Mumbai Indians, and winning the IPL trophy in 2009 with the Deccan Chargers. I have saved this information to your profile.


### Config -> thread_id = rohit_264 and user_name = rohit_45 (same as in previous config with thread_id = rohit_45)

In [9]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Hi! Can you tell me about my profile?"}]}, 
    config=config
)

print("Response:", result["messages"][-1].content)

c:\Users\gaura\OneDrive\Desktop\AI projects\agentic_ai_advanced\DeepAgents\deepagents\.venv\Lib\site-packages\deepagents\middleware\filesystem.py:1679: LangChainDeprecationWarning: Passing a callable (factory) as `backend` is deprecated and will be removed in deepagents==0.7.0. Pass a `BackendProtocol` instance directly instead (e.g. `StateBackend()`).
  backend = self._get_backend(request.runtime)  # ty: ignore[invalid-argument-type]
C:\Users\gaura\AppData\Local\Temp\ipykernel_26192\3982891131.py:18: LangChainDeprecationWarning: Passing `runtime` to `StateBackend` is deprecated and will be removed in deepagents==0.7.0. `StateBackend` now reads and writes state via `get_config()`. Use `StateBackend()` instead.
  default=StateBackend(rt),           # ephemeral working files
C:\Users\gaura\AppData\Local\Temp\ipykernel_26192\3982891131.py:20: LangChainDeprecationWarning: Passing `runtime` to `StoreBackend` is deprecated and will be removed in deepagents==0.7.0. `StoreBackend` now obtains 

Response: Your profile contains the following cricketing achievements:

*   Won 6 IPL trophies in career.
*   Won 5 cups with Mumbai Indians franchise as captain.
*   Won IPL trophy in 2009 as a player with Deccan Chargers.


## Chat isolation based on user_id (accessible across different threads for a user profile based on user_id)

In [10]:
from deepagents.backends import StoreBackend, CompositeBackend, StateBackend

# Pass the backend instance directly (the modern way)
agent = create_deep_agent(
    model=model,
    store=postgres_store,
    backend=CompositeBackend(
        default=StateBackend(), # Raw chat history and heavy tool dumps stay thread-scoped
        routes={
            "/memories/": StoreBackend( 
                store=postgres_store,
                # REMOVED thread_id so it persists globally for this user!
                namespace=lambda rt: (
                    rt.server_info.user.identity if rt.server_info 
                    else getattr(rt.context, "user_id", "default_local_user"),
                    "global_memories" 
                ), 
            )
        },
    ),
    system_prompt=(
        "You are a helpful assistant with persistent memory. "
        "Whenever the user shares personal information, preferences, or important facts, "
        "proactively save them to /memories/profile.txt so you can recall them later. "
        "Before answering questions about the user, read /memories/profile.txt first."
    ),
)

In [11]:
config_thread_1 = {
    "configurable": {"thread_id": "thread_11"}, # Thread A
    "context": {"user_id": "gaurav_45"}             # User Identity
}

In [12]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Hi! My name is Gaurav, I am a Research Engineer in an IT company. My responsibilities are designing and developing software solution with cutting edge technologies. My skills are full stack software developement using MERN stack, iOS, python, AI application developement along with LLMOps. Also, i have good understanding of security and networking.  I love solving real problems with efficient code. I am also learning about cloud computing and currently exploring AWS services."}]}, 
    config=config_thread_1
)

print("Response:", result["messages"][-1].content)

Response: /memories/profile.txt has been updated with your information. How can I help you today, Gaurav?


### Keep the user_id same but differ the thread in config

In [14]:
config_thread_2 = {
    "configurable": {"thread_id": "thread_22"}, # Thread B
    "context": {"user_id": "gaurav_45"}             #same User Identity
}

result = agent.invoke(
    {"messages": [{"role": "user", "content": "My favourite player is Rohit Sharma. He is an amazing cricketer and one of the best batsmen in the world."}]}, 
    config=config_thread_2
)

result


{'messages': [HumanMessage(content='My favourite player is Rohit Sharma. He is an amazing cricketer and one of the best batsmen in the world.', additional_kwargs={}, response_metadata={}, id='c8ff70e2-684a-4c3c-980c-320db96b6880'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-06-14T20:39:59.8623254Z', 'done': True, 'done_reason': 'stop', 'total_duration': 14306997100, 'load_duration': 439055500, 'prompt_eval_count': 5927, 'prompt_eval_duration': 788714400, 'eval_count': 177, 'eval_duration': 12811454700, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019ec7dc-a090-7d11-ace2-1713b753e3fe-0', tool_calls=[{'name': 'write_file', 'args': {'content': 'Favorite cricketer: Rohit Sharma. He is considered an amazing and world-class batsman.', 'file_path': '/memories/profile.txt'}, 'id': '47c172db-eea9-4a7f-86f2-1b18bbd306bb', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_

In [16]:
print("Response:", result["messages"][-1].content)

Response: /memories/profile.txt has been updated to include your favorite cricketer, Rohit Sharma.


In [15]:
result2 = agent.invoke(
    {"messages":[{"role": "user", "content": "Hi! what you know about me?"}]},
    config=config_thread_2
)

result2

{'messages': [HumanMessage(content='Hi! what you know about me?', additional_kwargs={}, response_metadata={}, id='3fbab852-f010-49c0-858c-5780caddefea'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-06-14T20:41:17.6492499Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4764531300, 'load_duration': 332436600, 'prompt_eval_count': 5913, 'prompt_eval_duration': 155026400, 'eval_count': 66, 'eval_duration': 4163359800, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019ec7dd-f5b2-7112-af5f-0626497df125-0', tool_calls=[{'name': 'read_file', 'args': {'file_path': '/memories/profile.txt'}, 'id': 'ded8babb-219c-4350-ae3f-7d3bad764c18', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 5913, 'output_tokens': 66, 'total_tokens': 5979}),
  ToolMessage(content='     1\tName: Gaurav\n     2\tProfession: Research Engineer in an IT company.\n     3\tResponsibilit

In [17]:
print("Response:", result2["messages"][-1].content)

Response: I know the following about you:

*   **Name:** Gaurav
*   **Profession:** Research Engineer in an IT company.
*   **Responsibilities:** Designing and developing software solutions with cutting-edge technologies.
*   **Skills:** Full stack software development (MERN stack), iOS, Python, AI application development, LLMOps.
*   **Knowledge Areas:** Good understanding of security and networking.
*   **Favorite cricketer:** Rohit Sharma, whom you consider an amazing and world-class batsman.
